# 023 — Planificación clásica con STRIPS y PDDL

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

En **planificación clásica** el estado es un conjunto de literales (mundo
cerrado) y cada acción **STRIPS** declara tres listas:

```text
aplicable(a, s)  ⟺  PRE(a) ⊆ s
RESULT(s, a)     =   (s \ DEL(a)) ∪ ADD(a)
```

Todo lo no mencionado en ADD/DEL persiste (suposición STRIPS — la solución
pragmática al problema del marco). **PDDL** estandariza la notación separando
*domain* (predicados + esquemas de acción con variables) y *problem* (objetos,
`:init`, `:goal`). Los planificadores modernos hacen búsqueda hacia adelante
con **heurísticas independientes del dominio** extraídas automáticamente
(delete relaxation → h_add, h_FF; landmarks), linaje que viene de **GraphPlan**
(grafo de niveles con mutexes). Decidir si existe un plan es PSPACE-completo:
las heurísticas desplazan la dificultad, no la eliminan.


## 🧮 El laboratorio como problema STRIPS

`run_lab("workflow", seed=23)` ejecuta una máquina de estados
`received → validated → waiting_approval → completed`. Es exactamente un
problema STRIPS de cadena:

```text
acción validar:   PRE {status_received}          ADD {status_validated}        DEL {status_received}
acción esperar:   PRE {status_validated}         ADD {status_waiting_approval} DEL {status_validated}
acción completar: PRE {status_waiting_approval}  ADD {status_completed, approved} DEL {status_waiting_approval}

init: {status_received}      goal: {status_completed}
```

El plan (único) es la secuencia de 3 acciones; la lista `events` del JSON es
su traza de ejecución. El motor incluso verifica las precondiciones en tiempo
de ejecución (`RuntimeError` si la transición es inválida) — el patrón
ejecutor/monitor en miniatura.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("workflow", seed=23)
show(result)


## Reflexión

1. El workflow del laboratorio tiene un único plan válido. ¿Qué propiedad de sus acciones (mira las listas PRE/DEL) elimina toda ramificación, y qué cambiaría si dos acciones fueran aplicables en el mismo estado?
2. La suposición STRIPS dice que lo no mencionado persiste. ¿Qué error concreto aparecería en el mundo de bloques si olvidas poner `libre(y)` en la lista DEL de `mover(b, x, y)`?
3. ¿Por qué 'el plan salió bien en el modelo' no garantiza nada sobre el mundo real, y qué componente (ausente en el laboratorio) convierte un planificador en un sistema utilizable?
